In [ ]:
import os
import yaml
import pytz
import requests
import numpy as np
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# ┌─────────────────────────────────────────────────────────┐
# │  1. SPORTS                                              │
# │  What sports are available?                             │
# │  GET /v3/sports                                         │
# │  Returns: Football, Basketball, Tennis, etc.            │
# └──────────────────┬──────────────────────────────────────┘
#                    │
#                    ↓
# ┌─────────────────────────────────────────────────────────┐
# │  2. LEAGUES                                             │
# │  What leagues exist in a sport?                         │
# │  GET /v3/leagues?sport=football                         │
# │  Returns: Premier League, La Liga, Bundesliga, etc.     │
# └──────────────────┬──────────────────────────────────────┘
#                    │
#                    ↓
# ┌─────────────────────────────────────────────────────────┐
# │  3. EVENTS                                              │
# │  What matches are coming up?                            │
# │  GET /v3/events?sport=football&league=premier-league    │
# │  Returns: Man Utd vs Liverpool, Chelsea vs Arsenal, etc.│
# └──────────────────┬──────────────────────────────────────┘
#                    │
#                    ↓
# ┌─────────────────────────────────────────────────────────┐
# │  4. ODDS                                                │
# │  What are the betting odds?                             │
# │  GET /v3/odds?eventId=123456&bookmakers=Bet365,Unibet   │
# │  Returns: Odds from multiple bookmakers                 │
# └─────────────────────────────────────────────────────────┘

In [31]:
API_BASE = "https://api.odds-api.io/v3"
BOOKMAKERS = "Bet365,Betfair Exchange"
TOTAL_MARKETS = {
"Bet365": ["Goals Over/Under", "Alternative Total Goals"],
"Betfair Exchange": ["Totals"],
}
TARGET_HDPS = [1.5, 2.5, 3.5]

def get_available_sport():
    """Check if football is available"""
    url = "https://api.odds-api.io/v3/sports"
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        return data
    except Exception as e:
        print(f"[ERROR] Failed to fetch league events: {e}")
        return []

def get_league_events(api_key, league="england-premier-league", limit=10):
    """Fetch list of upcoming events for a given league."""
    url = f"{API_BASE}/events?apiKey={api_key}&sport=football&league={league}&limit={limit}"
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        return data
    except Exception as e:
        print(f"[ERROR] Failed to fetch league events: {e}")
        return []


def get_event_odds(api_key, event_id):
    """Fetch odds data for a specific event."""
    url = f"{API_BASE}/odds?apiKey={api_key}&eventId={event_id}&bookmakers={BOOKMAKERS}"
    try:
        response = requests.get(url)
        response.raise_for_status()
        odds_data = response.json()
        return odds_data
    except Exception as e:
        print(f"[ERROR] Failed to fetch odds for event {event_id}: {e}")
        return {}


def extract_events_to_df(events):
    """Convert event list to DataFrame with timezone-aware match times."""
    perth_tz = pytz.timezone("Australia/Perth")
    rows = []

    for e in events:
        # Convert UTC timestamp ('Z' → '+00:00') to Perth local datetime
        raw_date = e.get("date")
        if raw_date:
            match_time_utc = datetime.fromisoformat(raw_date.replace("Z", "+00:00"))
            match_time_local = match_time_utc.astimezone(perth_tz)
        else:
            match_time_local = None
        rows.append({
            "event_id": e.get("id"),
            "date": match_time_local.strftime("%Y-%m-%d") if match_time_local else None,
            "home_team": e.get("home"),
            "away_team": e.get("away"),
            "competition": e.get("league", {}).get("name"),
            "match_time": match_time_local,  # timezone-aware datetime
            "odds_time": datetime.now(perth_tz).strftime("%Y-%m-%d %H:%M:%S")
        })
    return pd.DataFrame(rows)


def extract_totals(odds_data):
    """Extract Totals market from odds data."""
    totals = []
    for bookmaker, markets in odds_data.get("bookmakers", {}).items():
        allowed_markets = TOTAL_MARKETS.get(bookmaker, [])
        for market in markets:
            # Skip irrelevant markets
            if market.get("name") not in allowed_markets:
                continue
            for o in market.get("odds", []):
                hdp = o.get("hdp")
                if hdp not in TARGET_HDPS:
                    continue
                totals.append({
                    "bookmaker": bookmaker,
                    "market_name": market.get("name"),
                    "hdp": hdp,
                    "over_odds": o.get("over"),
                    "under_odds": o.get("under")
                })
    return totals

def extract_btts(odds_data):
    """Extract Both Teams to Score market from odds data."""
    btts = []
    for bookmaker, markets in odds_data.get("bookmakers", {}).items():
        for market in markets:
            if market.get("name") == "Both Teams to Score" or market.get("name") == "Both Teams To Score":
                for o in market.get("odds", []):
                        btts.append({
                            "bookmaker": bookmaker,
                            "yes": o.get("yes"),
                            "no": o.get("no"),
                        })
    return btts

def pivot_totals(df_totals):
    """Pivot totals DataFrame so each bookmaker × odds type × hdp becomes a column."""
    if df_totals.empty:
        return df_totals

    # Always make an explicit copy so we can safely modify columns
    df_totals = df_totals.copy()

    # Clean bookmaker names for column naming
    df_totals.loc[:, "bookmaker_clean"] = df_totals["bookmaker"].str.replace(" ", "_", regex=False)

    # Melt into long form (over/under values stacked)
    df_long = df_totals.melt(
        id_vars=[
            "event_id", "home_team", "away_team", "competition",
            "match_time", "bookmaker_clean", "hdp", "odds_time"
        ],
        value_vars=["over_odds", "under_odds"],
        var_name="odds_type",
        value_name="odds_value"
    )

    # Build clean column names like Bet365_over_odds_2.5
    df_long.loc[:, "col_name"] = (
        df_long["bookmaker_clean"] + "_" + df_long["odds_type"] + "_" + df_long["hdp"].astype(str)
    )

    # Pivot back to wide format
    df_pivot = df_long.pivot_table(
        index=["event_id", "home_team", "away_team", "competition", "match_time", "odds_time"],
        columns="col_name",
        values="odds_value",
        aggfunc="first"
    ).reset_index()

    # Flatten MultiIndex columns (remove hierarchy)
    df_pivot.columns.name = None

    return df_pivot


def ensure_dir(path):
    """Ensure directory exists."""
    os.makedirs(path, exist_ok=True)

def save_dataframe(df, name, folder="data/exports"):
    """Save a DataFrame with timestamp in the given folder."""
    ensure_dir(folder)
    date = datetime.now().strftime("%Y%m%d")
    filename = f"{name}_{date}.csv"
    filepath = os.path.join(folder, filename)
    df.to_csv(filepath, index=False)
    print(f"Saved: {filepath}")
    return filepath

def main(api_key):
    """Fetch EPL events, extract Totals (by line) and BTTS, and save separately."""
    events = get_league_events(api_key)
    if not events:
        print("No events found.")
        return pd.DataFrame(), pd.DataFrame()

    df_events = extract_events_to_df(events)

    totals_rows = []
    btts_rows = []

    for _, row in df_events.iterrows():
        event_id = row["event_id"]
        print(f"Fetching odds for {row['home_team']} vs {row['away_team']}")

        odds_data = get_event_odds(api_key, event_id)
        if not odds_data:
            continue

        # --- Totals ---
        totals = extract_totals(odds_data)
        for t in totals:
            totals_rows.append({
                "event_id": event_id,
                "home_team": row["home_team"],
                "away_team": row["away_team"],
                "competition": row["competition"],
                "match_time": row["match_time"],
                "bookmaker": t["bookmaker"],
                "hdp": t["hdp"],
                "over_odds": t["over_odds"],
                "under_odds": t["under_odds"],
                "odds_time": row["odds_time"]
            })

        # --- BTTS ---
        btts = extract_btts(odds_data)
        for b in btts:
            btts_rows.append({
                "event_id": event_id,
                "home_team": row["home_team"],
                "away_team": row["away_team"],
                "competition": row["competition"],
                "match_time": row["match_time"],
                "bookmaker": b["bookmaker"],
                "yes_odds": b["yes"],
                "no_odds": b["no"],
                "odds_time": row["odds_time"]
            })

    # --- Build DataFrames ---
    df_totals_all = pd.DataFrame(totals_rows)
    df_btts = pd.DataFrame(btts_rows)

    if not df_totals_all.empty:
        for hdp in [1.5, 2.5, 3.5]:
            df_hdp = df_totals_all[df_totals_all["hdp"] == hdp].copy()
            df_totals_pivoted = pivot_totals(df_hdp)
    
            if df_totals_pivoted.empty:
                print(f"No data for hdp={hdp}")
                continue
            
            # Define columns
            col_365_over = f"Bet365_over_odds_{hdp}"
            col_365_under = f"Bet365_under_odds_{hdp}"
            col_bf_over = f"Betfair_Exchange_over_odds_{hdp}"
            col_bf_under = f"Betfair_Exchange_under_odds_{hdp}"
    
            # Convert odds columns to numeric (coerce invalid strings to NaN)
            for col in [col_365_over, col_365_under, col_bf_over, col_bf_under]:
                df_totals_pivoted[col] = pd.to_numeric(df_totals_pivoted[col], errors="coerce")
    
            # --- Vectorized RPD calculation ---
            # Over RPD
            o1 = df_totals_pivoted[col_365_over]
            o2 = df_totals_pivoted[col_bf_over]
            df_totals_pivoted["Over RPD"] = np.where(
                o1.isna() | o2.isna(),
                np.nan,
                np.where(
                    o1 > o2,
                    1,
                    np.maximum(abs(o1 - o2) / ((o1 + o2) / 2) * 100, 1)
                )
            ).round(3)
    
            # Under RPD
            u1 = df_totals_pivoted[col_365_under]
            u2 = df_totals_pivoted[col_bf_under]
            df_totals_pivoted["Under RPD"] = np.where(
                u1.isna() | u2.isna(),
                np.nan,
                np.where(
                    u1 > u2,
                    1,
                    np.maximum(abs(u1 - u2) / ((u1 + u2) / 2) * 100, 1)
                )
            ).round(3)
    
            # Optionally, drop rows where both RPDs are NaN
            df_totals_pivoted = df_totals_pivoted.dropna(subset=["Over RPD", "Under RPD"], how="all")
    
            save_dataframe(df_totals_pivoted, f"totals_hdp_{hdp}", folder="data/exports/totals")
    else:
        print("No totals data found")


    # --- Save BTTS data ---
    if not df_btts.empty:
        save_dataframe(df_btts, "btts", folder="data/exports/btts")
    else:
        print("No BTTS data found")

    return df_totals_all, df_btts

# Example usage
api_key = os.getenv("ODDS_API_KEY")
df_totals, df_btts = main(api_key)

Fetching odds for Burnley FC vs Chelsea FC
Fetching odds for Fulham FC vs Sunderland AFC
Fetching odds for Wolverhampton Wanderers vs Crystal Palace
Fetching odds for Liverpool FC vs Nottingham Forest
Fetching odds for Manchester United vs Everton FC
Fetching odds for Brighton & Hove Albion vs Brentford FC
Fetching odds for AFC Bournemouth vs West Ham United
Fetching odds for Newcastle United vs Manchester City
Fetching odds for Leeds United vs Aston Villa
Fetching odds for Arsenal FC vs Tottenham Hotspur
Fetching odds for Manchester United vs Everton FC
Fetching odds for Sunderland AFC vs AFC Bournemouth
Fetching odds for Brentford FC vs Burnley FC
Fetching odds for Manchester City vs Leeds United
Fetching odds for Everton FC vs Newcastle United
Fetching odds for Tottenham Hotspur vs Fulham FC
Fetching odds for Crystal Palace vs Manchester United
Fetching odds for Nottingham Forest vs Brighton & Hove Albion
Fetching odds for Aston Villa vs Wolverhampton Wanderers
Fetching odds for Wes

In [30]:
df = pd.read_csv("data/exports/totals/totals_hdp_3.5_20251111.csv")
df

,event_id,home_team,away_team,competition,match_time,odds_time,Bet365_over_odds_3.5,Bet365_under_odds_3.5,Betfair_Exchange_over_odds_3.5,Betfair_Exchange_under_odds_3.5,Over RPD,Under RPD
0,61300733,AFC Bournemouth,West Ham United,England - Premier League,2025-11-22 23:00:00+08:00,2025-11-11 21:52:07,2.750,1.444,2.21,1.40,21.774,3.094
1,61300735,Arsenal FC,Tottenham Hotspur,England - Premier League,2025-11-24 00:30:00+08:00,2025-11-11 21:52:07,2.750,1.444,2.52,1.38,8.729,4.533
2,61300737,Brighton & Hove Albion,Brentford FC,England - Premier League,2025-11-22 23:00:00+08:00,2025-11-11 21:52:07,2.750,1.444,1.70,NaN,47.191,NaN
3,61300739,Burnley FC,Chelsea FC,England - Premier League,2025-11-22 20:30:00+08:00,2025-11-11 21:52:07,3.200,1.363,1.08,NaN,99.065,NaN
4,61300741,Fulham FC,Sunderland AFC,England - Premier League,2025-11-22 23:00:00+08:00,2025-11-11 21:52:07,3.750,1.285,3.71,1.21,1.072,6.012
5,61300743,Leeds United,Aston Villa,England - Premier League,2025-11-23 22:00:00+08:00,2025-11-11 21:52:07,3.750,1.285,1.09,NaN,109.917,NaN
6,61300745,Liverpool FC,Nottingham Forest,England - Premier League,2025-11-22 23:00:00+08:00,2025-11-11 21:52:07,2.375,1.571,2.02,1.47,16.155,6.643
7,61300749,Newcastle United,Manchester City,England - Premier League,2025-11-23 01:30:00+08:00,2025-11-11 21:52:07,2.625,1.500,2.18,1.41,18.522,6.186
8,61300751,Wolverhampton Wanderers,Crystal Palace,England - Premier League,2025-11-22 23:00:00+08:00,2025-11-11 21:52:07,3.500,1.300,2.85,1.26,20.472,3.125
9,63887879,Manchester United,Everton FC,England - Premier League,2025-11-25 04:00:00+08:00,2025-11-11 21:52:07,2.750,1.444,2.35,1.36,15.686,5.991


,event_id,home_team,away_team,competition,match_time,odds_time,Bet365_over_odds_3.5,Bet365_under_odds_3.5,Betfair_Exchange_over_odds_3.5,Betfair_Exchange_under_odds_3.5,Over RPD,Under RPD
0,61300733,AFC Bournemouth,West Ham United,England - Premier League,2025-11-22 23:00:00+08:00,2025-11-11 21:50:20,2.750,1.444,2.21,1.40,21.774,3.094
1,61300735,Arsenal FC,Tottenham Hotspur,England - Premier League,2025-11-24 00:30:00+08:00,2025-11-11 21:50:20,2.750,1.444,2.52,1.38,8.729,4.533
2,61300737,Brighton & Hove Albion,Brentford FC,England - Premier League,2025-11-22 23:00:00+08:00,2025-11-11 21:50:20,2.750,1.444,1.70,NaN,47.191,NaN
3,61300739,Burnley FC,Chelsea FC,England - Premier League,2025-11-22 20:30:00+08:00,2025-11-11 21:50:20,3.200,1.363,1.08,NaN,99.065,NaN
4,61300741,Fulham FC,Sunderland AFC,England - Premier League,2025-11-22 23:00:00+08:00,2025-11-11 21:50:20,3.750,1.285,3.71,1.21,1.072,6.012
5,61300743,Leeds United,Aston Villa,England - Premier League,2025-11-23 22:00:00+08:00,2025-11-11 21:50:20,3.750,1.285,1.09,NaN,109.917,NaN
6,61300745,Liverpool FC,Nottingham Forest,England - Premier League,2025-11-22 23:00:00+08:00,2025-11-11 21:50:20,2.375,1.571,2.02,1.47,16.155,6.643
7,61300749,Newcastle United,Manchester City,England - Premier League,2025-11-23 01:30:00+08:00,2025-11-11 21:50:20,2.625,1.500,2.18,1.41,18.522,6.186
8,61300751,Wolverhampton Wanderers,Crystal Palace,England - Premier League,2025-11-22 23:00:00+08:00,2025-11-11 21:50:20,3.500,1.300,2.85,1.26,20.472,3.125
9,63887879,Manchester United,Everton FC,England - Premier League,2025-11-25 04:00:00+08:00,2025-11-11 21:50:20,2.750,1.444,2.35,1.36,15.686,5.991


In [22]:
df_totals_pivoted = df_totals_pivoted.dropna()
df_totals_pivoted

,event_id,home_team,away_team,competition,match_time,odds_time,Bet365_over_odds_3.5,Bet365_under_odds_3.5,Betfair_Exchange_over_odds_3.5,Betfair_Exchange_under_odds_3.5
0,61300733,AFC Bournemouth,West Ham United,England - Premier League,2025-11-22 23:00:00+08:00,2025-11-11 21:37:24,2.750,1.444,2.21,1.40
1,61300735,Arsenal FC,Tottenham Hotspur,England - Premier League,2025-11-24 00:30:00+08:00,2025-11-11 21:37:24,2.750,1.444,2.52,1.38
2,61300737,Brighton & Hove Albion,Brentford FC,England - Premier League,2025-11-22 23:00:00+08:00,2025-11-11 21:37:24,2.750,1.444,1.70,N/A
3,61300739,Burnley FC,Chelsea FC,England - Premier League,2025-11-22 20:30:00+08:00,2025-11-11 21:37:24,3.200,1.363,1.08,N/A
4,61300741,Fulham FC,Sunderland AFC,England - Premier League,2025-11-22 23:00:00+08:00,2025-11-11 21:37:24,3.750,1.285,3.71,1.21
5,61300743,Leeds United,Aston Villa,England - Premier League,2025-11-23 22:00:00+08:00,2025-11-11 21:37:24,3.750,1.285,1.09,N/A
6,61300745,Liverpool FC,Nottingham Forest,England - Premier League,2025-11-22 23:00:00+08:00,2025-11-11 21:37:24,2.375,1.571,2.02,1.47
7,61300749,Newcastle United,Manchester City,England - Premier League,2025-11-23 01:30:00+08:00,2025-11-11 21:37:24,2.625,1.500,2.18,1.41
8,61300751,Wolverhampton Wanderers,Crystal Palace,England - Premier League,2025-11-22 23:00:00+08:00,2025-11-11 21:37:24,3.500,1.300,2.85,1.26
19,63887879,Manchester United,Everton FC,England - Premier League,2025-11-25 04:00:00+08:00,2025-11-11 21:37:24,2.750,1.444,2.35,1.36


In [23]:
pd.to_numeric(df_totals_pivoted['Betfair_Exchange_under_odds_3.5'], errors="coerce")

0     1.40
1     1.38
2      NaN
3      NaN
4     1.21
5      NaN
6     1.47
7     1.41
8     1.26
19    1.36
Name: Betfair_Exchange_under_odds_3.5, dtype: float64

In [5]:
df = pd.read_csv("data/exports/totals/totals_hdp_3.5_20251111.csv")
df[["home_team", "away_team", ]]

,home_team,away_team
0,AFC Bournemouth,West Ham United
1,Arsenal FC,Tottenham Hotspur
2,Brighton & Hove Albion,Brentford FC
3,Burnley FC,Chelsea FC
4,Fulham FC,Sunderland AFC
5,Leeds United,Aston Villa
6,Liverpool FC,Nottingham Forest
7,Newcastle United,Manchester City
8,Wolverhampton Wanderers,Crystal Palace
9,Aston Villa,Wolverhampton Wanderers
